# 花店情境課程：小練習解答範例集
Status: in_use  
Prereq: list/dict、函式、模組化基本、能獨立跑小專題  
Can-do checklist:  
- [ ] 能把需求拆成資料結構與操作函式
- [ ] 能用小測資驗證每個操作
- [ ] 能把程式整理成可重跑的主程式/模組


用途：集中記錄每節課的小練習解答範例，方便回顧與補缺。

## 01 回溫 + 資料結構建模（對應 01_flower_shop_warmup_modeling）

In [1]:
# 練習 1：新增一種花材到 inventory
inventory = {
    "玫瑰": {"cost": 15, "price": 30, "stock": 20},
    "百合": {"cost": 18, "price": 36, "stock": 12},
}

inventory["康乃馨"] = {"cost": 10, "price": 20, "stock": 25}

inventory

{'玫瑰': {'cost': 15, 'price': 30, 'stock': 20},
 '百合': {'cost': 18, 'price': 36, 'stock': 12},
 '康乃馨': {'cost': 10, 'price': 20, 'stock': 25}}

In [2]:
# 練習 2：sell_flower 回傳缺貨數量

def sell_flower(inv, name, qty):
    if name not in inv:
        return False, "查無品項", qty
    if inv[name]["stock"] >= qty:
        inv[name]["stock"] -= qty
        return True, "成交", 0
    shortage = qty - inv[name]["stock"]
    return False, "庫存不足", shortage

sell_flower(inventory, "玫瑰", 30)

(False, '庫存不足', 10)

In [3]:
# 練習 3：show_inventory

def show_inventory(inv):
    for name, info in inv.items():
        print(name, info["stock"])

show_inventory(inventory)

玫瑰 20
百合 12
康乃馨 25


## 02 訂單流程與例外處理（對應 02_flower_shop_orders_exceptions）

In [4]:
# 練習 1：訂單總價超過 500 打 9 折

def calc_total_with_discount(total):
    return total * 0.9 if total > 500 else total

calc_total_with_discount(620)

558.0

In [5]:
# 練習 2：部分可出貨清單

def partial_ship(inv, items):
    ok_items = []
    lack_items = []
    for name, qty in items:
        if name in inv and inv[name]["stock"] >= qty:
            ok_items.append((name, qty))
        else:
            lack_items.append((name, qty))
    return ok_items, lack_items

inventory = {"玫瑰": {"stock": 3}, "百合": {"stock": 1}}
partial_ship(inventory, [("玫瑰", 2), ("百合", 2)])

([('玫瑰', 2)], [('百合', 2)])

In [6]:
# 練習 3：訂單列表統計今日營收

def total_revenue(orders):
    return sum(o["total"] for o in orders)

orders = [
    {"id": "OD1", "total": 120},
    {"id": "OD2", "total": 260},
]

total_revenue(orders)

380

## 04 訂單模組完整化（對應 04_flower_shop_order_module）

In [ ]:
from dataclasses import dataclass

@dataclass
class Order:
    order_id: str
    items: list
    total: int = 0
    status: str = "created"
    remark: str = ""
    pickup: str = "自取"

In [ ]:
# 小練習 2：狀態切換

def update_status(order, status):
    order.status = status
    return order

order = Order("OD004", [("玫瑰", 1)], status="paid")
update_status(order, "preparing")
update_status(order, "done")
order

In [ ]:
# 小練習 3：新增訂單並計算 total
inventory = {"玫瑰": {"price": 30}, "百合": {"price": 36}}


def calc_total(inv, items):
    return sum(inv[name]["price"] * qty for name, qty in items)

order2 = Order("OD005", [("百合", 2)], remark="一般", pickup="外送")
order2.total = calc_total(inventory, order2.items)
order2

In [ ]:
# 小練習 4：修改標題字串

def print_order(order):
    print("=== 花店訂單 ===")
    print(f"訂單編號: {order.order_id}")
    print(f"狀態: {order.status}")
    print(f"取件方式: {order.pickup}")
    print(f"備註: {order.remark}")
    print("-" * 40)
    print(f"{'品項':<10} {'數量':>6}")
    print("-" * 40)
    for name, qty in order.items:
        print(f"{name:<10} {qty:>6}")
    print("-" * 40)
    print(f"總金額: {order.total}")
    print("=" * 40)

print_order(order2)

In [ ]:
# 小練習 5：統計各狀態訂單數量

def count_status(orders):
    result = {}
    for o in orders:
        result[o.status] = result.get(o.status, 0) + 1
    return result

orders = [
    Order("OD1", [], status="created"),
    Order("OD2", [], status="preparing"),
    Order("OD3", [], status="done"),
    Order("OD4", [], status="done"),
]

count_status(orders)

## 03 進貨/補貨 + 模組化（對應 03_flower_shop_procurement_module）

In [ ]:
from dataclasses import dataclass

@dataclass
class Item:
    name: str
    cost: int
    price: int

@dataclass
class Stock:
    item: Item
    qty: int

In [ ]:
# 小練習 1：建立百合 Item 與 Stock
lily = Item("百合", 18, 36)
lily_stock = Stock(lily, 12)
lily_stock

In [ ]:
# 小練習 2：新增康乃馨並補貨玫瑰

def add_new_item(inv, item, qty):
    if item.name in inv:
        return False
    inv[item.name] = Stock(item, qty)
    return True


def restock(inv, name, qty):
    if name in inv:
        inv[name].qty += qty
        return True
    return False

inventory = {}
add_new_item(inventory, Item("玫瑰", 15, 30), 20)
add_new_item(inventory, Item("康乃馨", 10, 20), 25)
restock(inventory, "玫瑰", 3)

inventory

In [ ]:
# 小練習 3：新增進貨單
@dataclass
class PurchaseOrder:
    supplier: str
    items: list

po2 = PurchaseOrder("NiceFlowers", [("康乃馨", 8), ("百合", 4)])
po2

In [ ]:
# 小練習 4：結構化打印（標題改為 今日庫存）

def print_inventory(inv):
    print("=== 今日庫存 ===")
    print("NAME	COST	PRICE	QTY")
    for name, stock in inv.items():
        print(f"{name}	{stock.item.cost}	{stock.item.price}	{stock.qty}")

print_inventory(inventory)


In [ ]:
# 課後小練習：remove_item / inventory_value / 顯示總成本

def remove_item(inv, name):
    if name in inv:
        del inv[name]
        return True
    return False


def inventory_value(inv):
    total = 0
    for stock in inv.values():
        total += stock.item.cost * stock.qty
    return total

inventory_value(inventory)

## 05 CSV 讀寫實作（對應 05_flower_shop_csv_io）

In [ ]:
# 練習 1：增加欄位 note 到 orders.csv
import csv

orders = [
    {"date": "2026-02-01", "order_id": "OD001", "total": 120, "note": ""},
    {"date": "2026-02-01", "order_id": "OD002", "total": 260, "note": "急件"},
]

with open("orders.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["date", "order_id", "total", "note"])
    writer.writeheader()
    for o in orders:
        writer.writerow(o)

In [ ]:
# 練習 2：列出某日訂單
import csv

def list_orders(date):
    result = []
    with open("orders.csv", "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["date"] == date:
                result.append(row)
    return result

list_orders("2026-02-01")

In [ ]:
# 練習 3：新增一筆訂單並更新當日營收
import csv

def add_order(order):
    with open("orders.csv", "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["date", "order_id", "total", "note"])
        writer.writerow(order)

add_order({"date": "2026-02-01", "order_id": "OD003", "total": 180, "note": ""})

# 計算當日營收

def revenue_by_date(date):
    total = 0
    with open("orders.csv", "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["date"] == date:
                total += int(row["total"])
    return total

revenue_by_date("2026-02-01")

## 06 JSON 讀寫實作（對應 06_flower_shop_json_io）

In [ ]:
# 練習 1：加入 note 欄位
orders = [
    {"id": "OD001", "items": [{"name": "玫瑰", "qty": 2}], "total": 60, "status": "paid"},
    {"id": "OD002", "items": [{"name": "百合", "qty": 1}], "total": 36, "status": "created"},
]

for o in orders:
    o["note"] = ""

orders

In [ ]:
# 練習 2：find_order

def find_order(orders, order_id):
    for o in orders:
        if o["id"] == order_id:
            return o
    return None

find_order(orders, "OD002")

In [ ]:
# 練習 3：寫回 JSON
import json

with open("orders.json", "w", encoding="utf-8") as f:
    json.dump(orders, f, ensure_ascii=False, indent=2)